# Pandas for ML/DL — Workshop Primer

**Goal:** Learn enough pandas to load, inspect, clean, and prepare data for a machine learning model.

**Topics:**
1. Series & DataFrame
2. Loading & saving data
3. Inspecting data
4. Selecting data
5. Handling missing values
6. Manipulating data
7. Preparing data for ML
8. Exercises

In [ ]:
import pandas as pd
import numpy as np
print(pd.__version__)

## 1. Series & DataFrame

| Type | What it is | NumPy analogy |
|------|-----------|---------------|
| `Series` | 1D labeled array (one column) | 1D `ndarray` with an index |
| `DataFrame` | 2D table (rows × columns) | 2D `ndarray` with row & column labels |

In [ ]:
# Series — a single column with an index
s = pd.Series([2.4, 3.1, 0.9, 1.7], index=["a", "b", "c", "d"])
print(s)
print("dtype:", s.dtype)

In [ ]:
# DataFrame — built from a dictionary {column_name: list_of_values}
data = {
    "age":      [25, 34, 28, 52, 40],
    "salary":   [50000, 80000, 62000, 120000, 95000],
    "dept":     ["IT", "Finance", "IT", "HR", "Finance"],
    "promoted": [False, True, False, True, True],
}
df = pd.DataFrame(data)
df

## 2. Loading & Saving Data

In practice you'll almost always load data from a file, not type it by hand.

In [ ]:
# Load from a URL (the most common real-world pattern)
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
titanic = pd.read_csv(url)
titanic.head()

In [ ]:
# Save to CSV (index=False avoids writing row numbers as a column)
# df.to_csv("my_data.csv", index=False)

# Other common formats:
# pd.read_excel("file.xlsx")     → Excel
# pd.read_json("file.json")      → JSON
# pd.read_parquet("file.parquet")→ Parquet (fast columnar format used in industry)

## 3. Inspecting Data

First thing you do with any new dataset.

In [ ]:
print("Shape:",  titanic.shape)          # (rows, cols)
print("Columns:", titanic.columns.tolist())

In [ ]:
titanic.head()    # first 5 rows

In [ ]:
titanic.tail(3)   # last 3 rows

In [ ]:
titanic.dtypes    # column data types — crucial before feeding into a model

In [ ]:
titanic.info()    # dtypes + non-null counts at a glance

In [ ]:
titanic.describe()    # stats for numeric columns

In [ ]:
# Count missing values per column — essential before any ML pipeline
titanic.isnull().sum()

## 4. Selecting Data

| Method | Select by | Example |
|--------|----------|---------|
| `df[col]` | column name | `df["Age"]` |
| `df[[c1, c2]]` | multiple columns | `df[["Age", "Fare"]]` |
| `.loc[row, col]` | **label** | `df.loc[0:4, "Age"]` |
| `.iloc[row, col]` | **position** | `df.iloc[0:5, 2]` |
| `df[mask]` | boolean condition | `df[df["Age"] > 30]` |

In [ ]:
# Single column → Series
titanic["Age"].head()

In [ ]:
# Multiple columns → DataFrame
titanic[["Name", "Age", "Survived"]].head()

In [ ]:
# loc: label-based (row label : column label)
titanic.loc[0:4, "Pclass":"Sex"]   # rows 0–4, columns from Pclass to Sex

In [ ]:
# iloc: position-based (row index : col index)
titanic.iloc[0:5, 1:4]             # rows 0–4, columns 1–3

In [ ]:
# Boolean indexing — filtering rows by condition
survivors = titanic[titanic["Survived"] == 1]
print("Survivors:", len(survivors))

# Combine conditions with & (and) or | (or)
young_survivors = titanic[(titanic["Survived"] == 1) & (titanic["Age"] < 18)]
print("Young survivors (<18):", len(young_survivors))

## 5. Handling Missing Values

Missing values break most ML models. You must handle them before training.

In [ ]:
# Which columns have missing values and how many?
missing = titanic.isnull().sum()
missing[missing > 0]

In [ ]:
# Strategy 1: Fill with a statistic (imputation) — good for numeric columns
titanic["Age"] = titanic["Age"].fillna(titanic["Age"].median())

# Strategy 2: Fill with a constant — good for categorical columns
titanic["Embarked"] = titanic["Embarked"].fillna("Unknown")

# Strategy 3: Drop rows/columns with too many missing values
# Cabin has ~77% missing — drop the column entirely
titanic = titanic.drop(columns=["Cabin"])

# Verify
titanic.isnull().sum()

## 6. Manipulating Data

Common transformations before feeding data into a model.

In [ ]:
# Add a new column derived from existing ones (feature engineering)
titanic["FamilySize"] = titanic["SibSp"] + titanic["Parch"] + 1
titanic[["SibSp", "Parch", "FamilySize"]].head()

In [ ]:
# Apply a function to a column
titanic["Fare_log"] = titanic["Fare"].apply(np.log1p)  # log-transform skewed data
titanic[["Fare", "Fare_log"]].head()

In [ ]:
# String methods on text columns
titanic["Sex_upper"] = titanic["Sex"].str.upper()
titanic[["Sex", "Sex_upper"]].head(3)

In [ ]:
# Drop columns you don't need
titanic = titanic.drop(columns=["Sex_upper", "Fare_log"])  # cleanup demo columns

# Rename a column
titanic = titanic.rename(columns={"Pclass": "PassengerClass"})
titanic.columns.tolist()

In [ ]:
# Groupby — aggregate by a category
titanic.groupby("Sex")[["Survived", "Age", "Fare"]].mean().round(2)

In [ ]:
# Value counts — distribution of a categorical column
titanic["Sex"].value_counts()

## 7. Preparing Data for ML

The final step before calling `model.fit()` — convert the DataFrame to numeric arrays.

In [ ]:
# Select features (X) and target (y)
features = ["PassengerClass", "Sex", "Age", "Fare", "FamilySize"]
target   = "Survived"

df_model = titanic[features + [target]].copy()
df_model.head()

In [ ]:
# Encode categorical column — ML models need numbers, not strings
df_model["Sex"] = df_model["Sex"].map({"male": 0, "female": 1})
df_model.head()

In [ ]:
# Confirm no missing values remain
print(df_model.isnull().sum())
print("\nAll dtypes numeric?", (df_model.dtypes != object).all())

In [ ]:
# Split into X (features) and y (label) — standard ML convention
X = df_model[features].values   # → NumPy array, shape (n_samples, n_features)
y = df_model[target].values     # → NumPy array, shape (n_samples,)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Sample X row:", X[0])
print("Sample y label:", y[0])

---
## 8. Exercises

Use the Titanic DataFrame (`titanic`) loaded above, or create your own.

---

### E1 — Create a DataFrame
Create a DataFrame from scratch with at least 5 rows and 4 columns representing students:
- `name` (string), `score` (int), `grade` (string: A/B/C), `passed` (bool)

Print its shape, dtypes, and a statistical summary.

In [ ]:
# Your code here


### E2 — Inspect the Titanic data
Using the `titanic` DataFrame:
- How many rows and columns does it have?
- Which columns have missing values and how many per column?
- What is the average `Fare` paid by passengers in each passenger class?

In [ ]:
# Your code here


### E3 — Selection
From the Titanic data:
- Select only the `Name`, `Age`, and `Survived` columns.
- Using `.iloc`, select rows 10–20 and the first 3 columns.
- Filter all **female** passengers who were in **1st class** and survived.

In [ ]:
# Your code here


### E4 — Missing values
Create this DataFrame with missing values and fix them:

```python
data = {
    "feature_1": [1.0, 2.0, np.nan, 4.0, 5.0],
    "feature_2": [10, np.nan, 30, np.nan, 50],
    "category":  ["A", "B", None, "A", "B"],
}
```

- Fill `feature_1` missing values with its **mean**.
- Fill `feature_2` missing values with its **median**.
- Fill `category` missing values with `"Unknown"`.
- Confirm no nulls remain.

In [ ]:
# Your code here


### E5 — Feature engineering
Using the Titanic data:
1. Create a boolean column `IsAlone` that is `True` when `FamilySize == 1`.
2. Create a column `AgeGroup` that bins age into: `child` (<12), `teen` (12–17), `adult` (18–59), `senior` (60+).  
   *Hint: use `pd.cut()` or `np.select()` or `.apply()` with a custom function.*
3. Show the survival rate for each `AgeGroup` using `.groupby()`.

In [ ]:
# Your code here


### E6 — Full ML prep pipeline
Starting from the Titanic data, build a clean feature matrix ready for a model:

1. Select columns: `PassengerClass`, `Sex`, `Age`, `Fare`, `FamilySize`, `IsAlone`, `Survived`.
2. Encode `Sex` as 0/1.
3. Check for any remaining nulls and handle them.
4. Split into `X` (all columns except `Survived`) and `y` (`Survived`).
5. Convert both to NumPy arrays and print their shapes.

At the end you should have `X.shape == (n, 6)` and `y.shape == (n,)`.

In [ ]:
# Your code here


### E7 — Bonus: Explore your own dataset
Load any CSV from the web with `pd.read_csv(url)` and answer:
1. What is the shape?
2. Are there any missing values?
3. What column would you use as the target (`y`) for a prediction task?
4. Which columns need cleaning or encoding before ML?

Some quick options:
- Heart disease: `https://raw.githubusercontent.com/datasciencedojo/datasets/master/heart.csv`
- Iris: `https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv`

In [ ]:
# Your code here
